# Lab 4 — Miscellaneous Python Topics
**Difficulty: Intermediate | ~25 min | Requires Lab 2**

In [1]:
# This lab uses only Python's built-in features, but this cell keeps the
# notebook runnable from a fresh kernel. It installs nothing extra.
!pip install --upgrade pip

## `match`/`case` pattern matching

Python 3.10+ `match`/`case` branches on the *shape* of a value, not just its equality — a cleaner alternative to a long `if`/`elif` chain.


In [2]:
# match/case: a modern alternative to long if/elif chains.
def classify_method(method):
    """Return a label for the given payment method using match/case."""
    match method:
        case "cash":
            return "Cash Payment"
        case "card":
            return "Card Payment"
        case "mobile":
            return "Mobile Wallet"
        case _:  # wildcard: anything not matched above
            return "Unknown Method"


# Classify each example sale and print its label.
print("--- Payment Method Classification ---")
for method, amount in [("cash", 25.00), ("card", 49.99), ("mobile", 15.00)]:
    label = classify_method(method)
    print(f"${amount:.2f} by {method:<8} -> {label}")


--- Payment Method Classification ---
$25.00 by cash     -> Cash Payment
$49.99 by card     -> Card Payment
$15.00 by mobile   -> Mobile Wallet


## `None` checks and defaults

`None` means "missing". Fill each missing field with a safe default so downstream code never trips over an empty value.


In [3]:
# None often means "missing"; fill each missing field with a safe default.
def apply_defaults(sale):
    """Fill in None fields with sensible defaults."""
    if sale.get("product") is None:
        sale["product"] = "SKU-0000-XX"
    if sale.get("price") is None:
        sale["price"] = 0.00
    if sale.get("qty") is None:
        sale["qty"] = 1
    if sale.get("method") is None:
        sale["method"] = "cash"
    return sale  # mutate in place and return for convenience


# Build an incomplete sale, fill the gaps, and show the result.
incomplete = {"product": None, "price": 5.99, "qty": None, "method": "card"}
fixed = apply_defaults(incomplete)
print(f"After defaults: {fixed}")


After defaults: {'product': 'SKU-0000-XX', 'price': 5.99, 'qty': 1, 'method': 'card'}


## Identity checks: `is` vs `==`

`is` compares identity (same object); `==` compares value and can be overridden. Always test for `None` with `is None`, never `== None`.


In [4]:
# None is a singleton; compare it with 'is', not with ==.
value = None
print(f"value is None:      {value is None}")
print(f"value == None:      {value == None}")
print(f"value is not None:  {value is not None}")
print()

# A custom class can make == lie, but 'is' always checks identity.
class Strict:
    def __eq__(self, other):
        return True  # pretends to equal everything

obj = Strict()
print(f"obj == None is {obj == None}  (misleading! )")  # == calls Strict.__eq__
print(f"obj is None is {obj is None}  (correct)")       # is checks identity only


value is None:      True
value == None:      True
value is not None:  False

obj == None is True  (misleading! )
obj is None is False  (correct)


## User input handling (simulated)

`input()` always returns a string. Strip whitespace and cast to the right type (`float`, `int`) before trusting it.


In [5]:
# Simulated cashier input (all strings from a terminal or form)
raw_code   = "SKU-4501-D7"
raw_price  = "29.95"
raw_qty    = "2"
raw_method = "card"

# Validate and cast: strip whitespace, convert to the right type.
product = raw_code.strip()
price   = float(raw_price)   # string -> number
qty     = int(raw_qty)       # string -> integer
method  = raw_method.strip().lower()

new_sale = {
    "product": product,
    "price": price,
    "qty": qty,
    "method": method,
    "date": None,
}
print(f"Parsed sale: {new_sale}")


Parsed sale: {'product': 'SKU-4501-D7', 'price': 29.95, 'qty': 2, 'method': 'card', 'date': None}


## `range()` and the `array` module

`range()` yields a lazy sequence of integers without a full list; `array` stores numbers in a compact typed layout.


In [6]:
# range(): a lazy sequence of integers, handy in for loops.
from array import array

print("--- range() ---")
for i in range(1, 6):
    print(i, end=" ")
print()

# A typed array keeps numeric data compact in memory ('i' = signed int).
daily_qty = array('i', [3, 1, 5, 2])
daily_qty.append(4)
print("Array:", list(daily_qty))
print("Sum:", sum(daily_qty), "| Max:", max(daily_qty))


--- range() ---
1 2 3 4 5 
Array: [3, 1, 5, 2, 4]
Sum: 15 | Max: 5


## Working with dates

`datetime.date.today()` returns today's date; `strftime` format codes control how it displays.


In [7]:
# date.today() captures the real current date, no arguments needed.
from datetime import date

today = date.today()
new_sale["date"] = today  # stamp the earlier parsed sale with today's date

print(f"--- Today's Date ---")
print(f"{today}")
print(f"Formatted: {today.strftime('%B %d, %Y')}")  # e.g. September 07, 2026
print(f"Month/Year: {today.strftime('%m/%Y')}")


--- Today's Date ---
2026-09-08
Formatted: September 08, 2026
Month/Year: 09/2026


## Regular expressions

A regex like `SKU-\d{4}-[A-Z]{2}` validates text patterns in one declarative expression, which is far shorter than manual checks.


In [8]:
# re.compile pre-parses the pattern once for reuse and speed.
import re

SKU_PATTERN = re.compile(r"SKU-\d{4}-[A-Z]{2}")

print("--- Product Code Validation ---")
for code in ["SKU-1001-A1", "SKU-2045-B2", "SKU-0310-C3", "INVALID-XYZ"]:
    valid = "Valid" if SKU_PATTERN.fullmatch(code) else "Invalid (expected SKU-NNNN-LL)"
    print(f"{code:<16} ->  {valid}")


--- Product Code Validation ---
SKU-1001-A1      ->  Invalid (expected SKU-NNNN-LL)
SKU-2045-B2      ->  Invalid (expected SKU-NNNN-LL)
SKU-0310-C3      ->  Invalid (expected SKU-NNNN-LL)
INVALID-XYZ      ->  Invalid (expected SKU-NNNN-LL)


## Seed data and combining records

Keep the running dataset as a list of dicts, then `append` new records so every later report sees the full history.


In [9]:
# Our running dataset: a list of sale dictionaries.
existing_sales = [
    {"product": "SKU-1001-A1", "price": 12.50, "qty": 3, "method": "cash",  "date": date(2026, 8, 28)},
    {"product": "SKU-2045-B2", "price": 49.99, "qty": 1, "method": "card",  "date": date(2026, 8, 29)},
    {"product": "SKU-0310-C3", "price":  7.25, "qty": 5, "method": "cash",  "date": date(2026, 8, 30)},
]

existing_sales.append(new_sale)  # add the sale we parsed in an earlier cell
print(f"Total sales records: {len(existing_sales)}")


Total sales records: 4


## String formatting: f-strings

f-strings put values directly inside a string — Python's modern default, ideal for building readable receipts.


In [10]:
# F-string formatting: values go directly inside the string.
def print_receipt_fstring(sales):
    print("--- Full Receipt (f-string) ---")
    for s in sales:
        total = s["price"] * s["qty"]
        line = f"| {s['product']:<16} | {s['qty']} x ${s['price']:.2f} | ${total:.2f} | {s['method']:<6} | {s['date']} |"
        print(line)


print_receipt_fstring(existing_sales)


--- Full Receipt (f-string) ---
| SKU-1001-A1      | 3 x $12.50 | $37.50 | cash   | 2026-08-28 |
| SKU-2045-B2      | 1 x $49.99 | $49.99 | card   | 2026-08-29 |
| SKU-0310-C3      | 5 x $7.25 | $36.25 | cash   | 2026-08-30 |
| SKU-4501-D7      | 2 x $29.95 | $59.90 | card   | 2026-09-08 |


## String formatting: `.format()`

`.`format()` fills named `{}` placeholders — useful for template strings or where the format string is reused.


In [11]:
# .format(): placeholders {} are filled by named values.
def print_receipt_format(sales):
    print("--- Full Receipt (.format()) ---")
    for s in sales:
        total = s["price"] * s["qty"]
        line = "| {product:<16} | {qty} x ${price:.2f} | ${total:.2f} | {method:<6} | {date} |"
        print(line.format(product=s["product"], qty=s["qty"],
                          price=s["price"], total=total,
                          method=s["method"], date=s["date"]))


print_receipt_format(existing_sales)


--- Full Receipt (.format()) ---
| SKU-1001-A1      | 3 x $12.50 | $37.50 | cash   | 2026-08-28 |
| SKU-2045-B2      | 1 x $49.99 | $49.99 | card   | 2026-08-29 |
| SKU-0310-C3      | 5 x $7.25 | $36.25 | cash   | 2026-08-30 |
| SKU-4501-D7      | 2 x $29.95 | $59.90 | card   | 2026-09-08 |


## String formatting: `%`-formatting

The oldest style uses `%` placeholders; you'll meet it in legacy code, so learn to read and write it.


In [12]:
# %-formatting: the oldest style, using % placeholders and a tuple.
def print_receipt_percent(sales):
    print("--- Full Receipt (%-formatting) ---")
    for s in sales:
        total = s["price"] * s["qty"]
        line = "| %-16s | %d x $%.2f | $%.2f | %-6s | %s |"
        print(line % (s["product"], s["qty"], s["price"], total, s["method"], s["date"]))


print_receipt_percent(existing_sales)


--- Full Receipt (%-formatting) ---
| SKU-1001-A1      | 3 x $12.50 | $37.50 | cash   | 2026-08-28 |
| SKU-2045-B2      | 1 x $49.99 | $49.99 | card   | 2026-08-29 |
| SKU-0310-C3      | 5 x $7.25 | $36.25 | cash   | 2026-08-30 |
| SKU-4501-D7      | 2 x $29.95 | $59.90 | card   | 2026-09-08 |


## Monthly summary

Filter the records to one month, sum revenue, and guard the average against division by zero.


In [13]:
# Summarize August sales from the receipt data.
target_month = 8
target_year  = 2026
month_sales  = [s for s in existing_sales if s["date"].month == target_month]
total_rev    = sum(s["price"] * s["qty"] for s in month_sales)
avg_sale     = total_rev / len(month_sales) if month_sales else 0  # avoid division by 0

print("--- Monthly Summary ({} sales in {} {}---".format(
    len(month_sales),
    date(target_year, target_month, 1).strftime("%b"),
    target_year))
print("Total revenue: $%.2f" % total_rev)
print("Average sale:  $%.2f" % avg_sale)


--- Monthly Summary (3 sales in Aug 2026---
Total revenue: $123.74
Average sale:  $41.25
